In [5]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

## Hyperparameters

In [6]:
VOCAB_SIZE  = 30000
SEQ_LEN     = 64   # context window
D_MODEL     = 128   # embedding / hidden dim
N_HEADS     = 8     # attention heads  (D_MODEL must be divisible by N_HEADS)
D_FF        = 512   # feed-forward inner dim
N_LAYERS    = 2     # transformer blocks
DROPOUT     = 0.1

BATCH_SIZE  = 64
LR          = 3e-4
EPOCHS      = 5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")

device: cuda


## Data loading

In [7]:
TRAIN_BIN = Path.cwd() / "data" / "train.bin"

token_ids = np.fromfile(TRAIN_BIN, dtype=np.uint16).astype(np.int64)
print(f"total tokens: {len(token_ids):,}")

class TokenDataset(Dataset):
    def __init__(self, ids, seq_len):
        self.ids = torch.tensor(ids, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.ids) - self.seq_len

    def __getitem__(self, idx):
        x = self.ids[idx : idx + self.seq_len]
        y = self.ids[idx + 1 : idx + self.seq_len + 1]
        return x, y

dataset    = TokenDataset(token_ids, SEQ_LEN)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print(f"batches per epoch: {len(dataloader)}")

total tokens: 136,633
batches per epoch: 2133


## Sinusoidal Positional Encoding

In [8]:
class SinusoidalPE(nn.Module):
    """Fixed sinusoidal positional encoding.
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model: int, max_len: int = 4096, dropout: float = 0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)           # (max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1)     # (max_len, 1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)) # (d_model/2,)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        pe = pe.unsqueeze(0)                         # (1, max_len, d_model)
        self.register_buffer("pe", pe)               # not a learnable param

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, D)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

## Multi-Head Attention

In [9]:
class MultiHeadAttention(nn.Module):
    """Scaled dot-product multi-head self-attention with causal mask."""

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        self.scale   = self.d_head ** -0.5

        # fused QKV projection (saves 2 matmuls vs three separate linears)
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj  = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        B, T, D = x.shape

        # Project and split into Q, K, V  →  each (B, T, D)
        qkv = self.qkv_proj(x)                          # (B, T, 3D)
        q, k, v = qkv.split(D, dim=-1)

        # Reshape to (B, n_heads, T, d_head)
        def split_heads(t):
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        q, k, v = split_heads(q), split_heads(k), split_heads(v)

        # Scaled dot-product attention scores  (B, H, T, T)
        attn = (q @ k.transpose(-2, -1)) * self.scale

        # Causal mask: positions can only attend to earlier positions
        if mask is None:
            mask = torch.tril(torch.ones(T, T, device=x.device)).bool()
        attn = attn.masked_fill(~mask, float("-inf"))

        attn = F.softmax(attn, dim=-1)
        attn = self.attn_drop(attn)

        # Weighted sum of values  (B, H, T, d_head) → (B, T, D)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.out_proj(out)

## Transformer Block (MHA + FFN)

In [10]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """Pre-norm transformer block: LayerNorm → MHA → residual, LayerNorm → FFN → residual."""

    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.0):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ln2  = nn.LayerNorm(d_model)
        self.ffn  = FeedForward(d_model, d_ff, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

## Full Language Model

In [ ]:
class MHATf(nn.Module):
    """Token embedding + sinusoidal PE + N transformer blocks + LM head."""

    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        n_heads: int,
        d_ff: int,
        n_layers: int,
        seq_len: int,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_enc = SinusoidalPE(d_model, max_len=seq_len, dropout=dropout)
        self.blocks  = nn.Sequential(*[TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.ln_f    = nn.LayerNorm(d_model)            # final layer norm
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # weight tying: embed and unembed share weights (standard practice)
        self.lm_head.weight = self.tok_emb.weight

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        # idx: (B, T)
        x = self.tok_emb(idx)    # (B, T, D)
        x = self.pos_enc(x)      # adds sinusoidal PE in-place
        x = self.blocks(x)       # 2 transformer blocks
        x = self.ln_f(x)
        return self.lm_head(x)   # (B, T, vocab_size)

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int, temperature: float = 1.0, top_k: int = None):
        """Autoregressive generation."""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -SEQ_LEN:]                      # crop to context window
            logits   = self(idx_cond)[:, -1, :]               # last-position logits
            logits   = logits / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs  = F.softmax(logits, dim=-1)
            next_t = torch.multinomial(probs, num_samples=1)
            idx    = torch.cat([idx, next_t], dim=1)
        return idx


model = MHATf(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    d_ff=D_FF,
    n_layers=N_LAYERS,
    seq_len=SEQ_LEN,
    dropout=DROPOUT,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params:,}")
print(model)

parameters: 4,235,776
ShallowSeek(
  (tok_emb): Embedding(30000, 128)
  (pos_enc): SinusoidalPE(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (blocks): Sequential(
    (0): TransformerBlock(
      (ln1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (attn): MultiHeadAttention(
        (qkv_proj): Linear(in_features=128, out_features=384, bias=False)
        (out_proj): Linear(in_features=128, out_features=128, bias=False)
        (attn_drop): Dropout(p=0.1, inplace=False)
      )
      (ln2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (ffn): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=128, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.1, inplace=False)
          (3): Linear(in_features=512, out_features=128, bias=True)
          (4): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (1): TransformerBlock(
      (ln1): LayerNorm((128,), eps=1e-05, elementwise_affi

## Training

In [12]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(dataloader))

def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0.0
    for step, (x, y) in enumerate(loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)                              # (B, T, V)
        loss   = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        if (step + 1) % 50 == 0:
            avg = total_loss / (step + 1)
            ppl = math.exp(avg)
            print(f"  step {step+1:4d}/{len(loader)}  loss={avg:.4f}  ppl={ppl:.1f}")

    return total_loss / len(loader)

for epoch in range(1, EPOCHS + 1):
    avg_loss = train_epoch(model, dataloader, optimizer, scheduler)
    print(f"[epoch {epoch}] loss={avg_loss:.4f}  ppl={math.exp(avg_loss):.1f}")

  step   50/2133  loss=9.0808  ppl=8785.4
  step  100/2133  loss=8.1285  ppl=3389.9
  step  150/2133  loss=7.5817  ppl=1961.9
  step  200/2133  loss=7.2933  ppl=1470.4
  step  250/2133  loss=7.1164  ppl=1232.0
  step  300/2133  loss=6.9983  ppl=1094.8
  step  350/2133  loss=6.9110  ppl=1003.3
  step  400/2133  loss=6.8447  ppl=938.9
  step  450/2133  loss=6.7939  ppl=892.4
  step  500/2133  loss=6.7538  ppl=857.3
  step  550/2133  loss=6.7204  ppl=829.2
  step  600/2133  loss=6.6921  ppl=806.0
  step  650/2133  loss=6.6685  ppl=787.2
  step  700/2133  loss=6.6483  ppl=771.4
  step  750/2133  loss=6.6308  ppl=758.1
  step  800/2133  loss=6.6158  ppl=746.8
  step  850/2133  loss=6.6025  ppl=737.0
  step  900/2133  loss=6.5901  ppl=727.8
  step  950/2133  loss=6.5792  ppl=719.9
  step 1000/2133  loss=6.5697  ppl=713.1
  step 1050/2133  loss=6.5606  ppl=706.7
  step 1100/2133  loss=6.5521  ppl=700.7
  step 1150/2133  loss=6.5441  ppl=695.2
  step 1200/2133  loss=6.5348  ppl=688.7
  step 12

## Text Generation

In [13]:
from tokenizers import Tokenizer

TOKENIZER_PATH = Path.cwd() / "models" / "tokenizer.json"
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

def generate_text(prompt: str, max_new_tokens: int = 100, temperature: float = 0.8, top_k: int = 50):
    model.eval()
    enc    = tokenizer.encode(prompt)
    ids    = torch.tensor([enc.ids], dtype=torch.long, device=DEVICE)
    out    = model.generate(ids, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    tokens = out[0].tolist()
    return tokenizer.decode(tokens)

prompt = "The king and the queen"
print(generate_text(prompt, max_new_tokens=80, temperature=0.9, top_k=40))

 The king and the queen was a case to this man's-room.

"What does?" He cried, "No. You have been a little very clear, it is not as you deduce to come to me. But there has been, you have the use, Doctor. We will make his father all."

" 'Oh, when you is a good," said he. "Not on the office


In [15]:
# Save model checkpoint
CKPT_PATH = Path.cwd() / "models" / "baseline_mha.pt"
CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    "model_state": model.state_dict(),
    "config": dict(
        vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_heads=N_HEADS,
        d_ff=D_FF, n_layers=N_LAYERS, seq_len=SEQ_LEN,
    ),
}, CKPT_PATH)